<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/test_image_RAG/Image_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q huggingface_hub langchain_core langchain_huggingface torch tiktoken datasets "chonkie[semantic]" chonkie chromadb "bitsandbytes>=0.46.1" pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.4/386.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/

In [2]:
import os
import re
import io
import torch
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv
from huggingface_hub import login, whoami
from datasets import load_dataset

In [3]:
hf_token = ""

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

login(token=hf_token)

print(whoami())

{'type': 'user', 'id': '6aab8629f4552c1f3e73bd1f', 'name': 'tdgbcfxdzr', 'fullname': 'Bao Tran', 'isPro': False, 'avatarUrl': '/avatars/6dfa5ced89e7d43ce0c793d6a070f4bd.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'MedGemma', 'role': 'fineGrained', 'createdAt': '2026-09-22T03:51:21.267Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6aab8629f4552c1f3e73bd1f', 'type': 'user', 'name': 'tdgbcfxdzr'}, 'permissions': ['repo.content.read']}]}}}}


In [4]:
repo_id = "choisongjun/Tropical_Diseases_Repo_1.6"

dataset_dict = load_dataset(repo_id, token=hf_token)
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"Train cases: {len(train_df)} | Test cases: {len(test_df)} | Diseases ({len(DISEASE_LABELS)}): {DISEASE_LABELS}")
train_df.head()

README.md:   0%|          | 0.00/573 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.59MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  671kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/83 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12 [00:00<?, ? examples/s]

Train cases: 83 | Test cases: 12 | Diseases (8): ['dengue', 'leptospirosis', 'malaria', 'melioidosis', 'neurocysticercosis', 'scrub_typhus', 'typhoid', 'zika']


,case_id,case_text,disease,images
0,PMC10870101_01,A 51-year-old female patient presented to our ...,dengue,[{'image_path': {'bytes': b'RIFF\xa4\x0e\x00\x...
1,PMC6417330_01,A 24-year-old female underwent transcranial su...,dengue,[{'image_path': {'bytes': b'RIFFBM\x00\x00WEBP...
2,PMC10876100_02,"A 43-year-old man, a resident of Uttar Pradesh...",dengue,[{'image_path': {'bytes': b'RIFF`7\x00\x00WEBP...
3,PMC10932498_01,A 17 year old male patient with no previous co...,dengue,[{'image_path': {'bytes': b'RIFF\xe85\x00\x00W...
4,PMC10062082_01,A 28-year-old male presented with complaint of...,dengue,[{'image_path': {'bytes': b'RIFF\xe6\x1b\x00\x...


Image Extractor and format definition

In [5]:
def extract_pil_images(image_field):

    images = []
    if not image_field.any():
        return images

    for item in image_field:
        if isinstance(item, dict):
            # Output raw image bytes into PIL Image objects of RGB format
            img_path_data = item.get("image_path", {})
            if isinstance(img_path_data, dict) and "bytes" in img_path_data:
                img_bytes = img_path_data["bytes"]
                img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                images.append(img)
            elif "bytes" in item:
                img = Image.open(io.BytesIO(item["bytes"])).convert("RGB")
                images.append(img)
        # Add pre-loaded PIL Image objects
        elif isinstance(item, Image.Image):
            images.append(item.convert("RGB"))
        # Handle local file paths
        elif isinstance(item, str) and os.path.exists(item):
            images.append(Image.open(item).convert("RGB"))

    return images

# Test image extraction on the first case from the train dataset
sample_images = extract_pil_images(train_df.iloc[0]["images"])
print(f"Extracted {len(sample_images)} image(s) from case {train_df.iloc[0]['case_id']}.")

Extracted 9 image(s) from case PMC10870101_01.


Text Cleaning & Semantic Chunking functions

In [6]:
from chonkie import (
    SemanticChunker,
    RecursiveRules,
    RecursiveLevel,
    OverlapRefinery,
)

rules = RecursiveRules(levels=[
    RecursiveLevel(delimiters=["\n\n"], include_delim="prev"),
    RecursiveLevel(delimiters=["\n"], include_delim="prev"),
    RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),
    RecursiveLevel(whitespace=True),
])

semantic_chunker = SemanticChunker(
    threshold=0.7,
    chunk_size=400,
    similarity_window=3,
    min_sentences_per_chunk=2,
    skip_window=0,
)

overlap_refinery = OverlapRefinery(
    tokenizer="character",
    context_size=120,
    mode="recursive",
    rules=rules,
    method="prefix",
    merge=True,
)

In [7]:
# Chunk functions
def clean_text(text: str) -> str:
    text = text.replace("et al.", "et al")
    text = re.sub(r"\[?(?:<[^>]+\.webp>)+(?:-\d+)?\]?", "", text)
    return text

def perform_semantic_chunking(text):
    chunks = semantic_chunker.chunk(clean_text(text))
    return [c.text for c in overlap_refinery(chunks)]

Build Vector store

In [8]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
import chromadb
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/colab_projects/VGU_WS26_Project/chroma_db")

In [10]:
COLLECTION_NAME = "tropical_disease_cases"

def build_collection(records, name, rebuild=False):
    if rebuild:
        try:
            chroma_client.delete_collection(name)
        except Exception:
            pass

    collection = chroma_client.get_or_create_collection(name)

    batch = chroma_client.get_max_batch_size()
    for start in range(0, len(records), batch):
        part = records[start:start + batch]
        collection.upsert(
            ids=[f"chunk_{start + i}" for i in range(len(part))],
            documents=[r["document"] for r in part],
            metadatas=[r["metadata"] for r in part],
        )

    print(f"{name}: built {collection.count()} chunks")
    return collection

def collection_ready(name):
  try:
      collection = chroma_client.get_collection(name)
  except Exception:
      return False
  return collection.count() > 0

# Iterate train dataset and chunk semantically
def make_records(df, chunk_fn):
    records = []
    for row in df.itertuples():
        chunks = chunk_fn(row.case_text)
        for chunk_id, chunk in enumerate(chunks):
            records.append({
                "document": chunk,
                "metadata": {
                    "case_id": row.case_id,
                    "disease": row.disease,
                    "chunk_id": chunk_id,
                    "total_chunks": len(chunks),
                },
            })
    return records

records = make_records(train_df, perform_semantic_chunking)

In [11]:
if collection_ready(COLLECTION_NAME):
    collection = chroma_client.get_collection(COLLECTION_NAME)
    print("Reusing existing collection")
else:
    records = make_records(train_df, perform_semantic_chunking)
    collections = build_collection(records, COLLECTION_NAME, rebuild=True)

print(f"ChromaDB collection '{COLLECTION_NAME}' created with {collection.count()} chunks.")

Reusing existing collection
ChromaDB collection 'tropical_disease_cases' created with 599 chunks.


Retrieval Function in RAG

In [12]:
# Queries ChromaDB at chunk level and aggregates top cases
def retrieve_cases(collection, query_text, k = 3, search_k = 20):
    res = collection.query(query_texts=[query_text], n_results=search_k)

    best_by_case = {}
    for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
        case_id = meta["case_id"]
        if case_id not in best_by_case or dist < best_by_case[case_id]["distance"]:
            best_by_case[case_id] = {
                "case_id": case_id,
                "disease": meta["disease"],
                "distance": dist,
            }

    return sorted(best_by_case.values(), key=lambda r: r["distance"])[:k]

# Look up the full case text for each retrieved case
def attach_case_text(hits, df):
    text_by_id = df.set_index("case_id")["case_text"]
    return [{**hit, "text": text_by_id.loc[hit["case_id"]]} for hit in hits]

Load MedGemma Model

In [13]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(model_id, token=hf_token)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

print("MedGemma successfully loaded.")

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

MedGemma successfully loaded.


In [14]:
SYSTEM_PROMPT = (
    "You are a careful clinical assistant. Use ONLY the retrieved case reports below as evidence.\n"
    "You MUST answer with exactly one disease name, copied verbatim from this list, and nothing else:\n"
    + "\n".join(f"- {name}" for name in DISEASE_LABELS)
)

def diagnose_image_rag(patient_text, patient_image = None, k = 3, max_new_tokens = 32):
    hits = retrieve_cases(collection, query_text=patient_text, k=k)
    retrieved_cases = attach_case_text(hits, train_df)

    # Combine retrieved cases
    context_str = "\n\n".join(
        f"Reference Case #{i+1} (Disease: {res['disease']}): {res['text'][:1000]}..."
        for i, res in enumerate(retrieved_cases)
    )

    # Join combined cases with patient's description
    user_prompt = (
        f"Retrieved Context Cases:\n{context_str}\n\n"
        f"Patient Clinical Presentation:\n{patient_text}\n\n"
        f"Which disease from the allowed list does this patient have?"
    )

    # Format multimodal payload
    user_content = []
    if patient_image is not None:
        user_content.append({"type": "image", "image": patient_image})
    user_content.append({"type": "text", "text": user_prompt})

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

    # Response generation
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    response = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return {
        "diagnosis": response,
        "retrieved_cases": retrieved_cases
    }

Run Image RAG Test Demo on the first case of the test dataset

In [15]:
# Get info from the first case
test_sample = test_df.iloc[0]
test_id = test_sample["case_id"]
test_text = test_sample["case_text"]
gt_disease = test_sample["disease"]
test_images = extract_pil_images(test_sample["images"])

query_img = test_images[0] if len(test_images) > 0 else None

print(f"Test patient ID: {test_id}")
print(f"Patient's actual disease: {gt_disease}")
print(f"Patient's image available: {query_img is not None}\n")

# Diagnose using a part of case text and images
rag_output = diagnose_image_rag(
    patient_text=test_text[:1200],
    patient_image=query_img,
    k=3
)

print("Retrieved cases:")
for r in rag_output["retrieved_cases"]:
    print(f"- Case ID: {r['case_id']} | Disease: {r['disease']} | Distance: {r['distance']:.3f}")

print("\nModel diagnosis:", rag_output["diagnosis"])

Test patient ID: PMC4750307_04
Patient's actual disease: dengue
Patient's image available: True



/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 30.1MiB/s]


RETRIEVED CASES:
- Case ID: PMC4750307_02 | Disease: dengue | Distance: 0.262
- Case ID: PMC4750307_05 | Disease: dengue | Distance: 0.270
- Case ID: PMC10876100_02 | Disease: dengue | Distance: 0.625

MODEL DIAGNOSIS: dengue
